
MemCtrl - Notebook 1: Load, Clean & Balance Data
=================================================

This notebook:
1. Loads your existing labeled dataset
2. CLEANS data quality issues (NEW!)
3. Fixes obvious label errors (NEW!)
4. Optionally applies merging/balancing
5. Saves final dataset for training

Author: Kamala
Date: December 2024



In [ ]:
#===============================================================================
# CELL 1: Setup
#===============================================================================

print("="*80)
print("MEMCTRL NOTEBOOK 1: LOAD, CLEAN & BALANCE DATA")
print("="*80)

from google.colab import drive
drive.mount('/content/drive')

import pandas as pd
import numpy as np
import json
from pathlib import Path
import re
import warnings
warnings.filterwarnings('ignore')

print("\n✓ Setup complete")

MEMCTRL NOTEBOOK 1: LOAD, CLEAN & BALANCE DATA
Mounted at /content/drive

✓ Setup complete


In [ ]:
CONFIG = {
    # Input: Your existing labeled data
    'input_data': '/content/drive/MyDrive/memctrl_data/labeled_dataset.parquet',

    # Output directory
    'output_dir': '/content/drive/MyDrive/memctrl_data',

    # CLEANING SETTINGS (NEW!)
    'apply_cleaning': True,           # Remove bad samples
    'min_prompt_length': 20,          # Remove ultra-short prompts
    'remove_meta_prompts': True,      # Remove jailbreak/safety warnings
    'remove_non_english': True,       # Remove non-English text
    'apply_quick_fixes': True,        # Fix obvious label errors

    # BALANCING SETTINGS
    'apply_merging': False,           # Set True to merge classes
    'max_samples_per_class': 1500,    # Cap dominant classes
    'min_samples_per_class': 100,     # Remove classes below this

    # Random seed
    'seed': 42,
}

print("\n📋 CONFIGURATION:")
print(f"  Apply cleaning: {CONFIG['apply_cleaning']}")
print(f"  Apply merging: {CONFIG['apply_merging']}")
print(f"  Min prompt length: {CONFIG['min_prompt_length']}")


📋 CONFIGURATION:
  Apply cleaning: True
  Apply merging: False
  Min prompt length: 20


In [ ]:
#===============================================================================
# CELL 3: Load Your Existing Labeled Data
#===============================================================================

print("\n" + "="*80)
print("LOADING YOUR EXISTING LABELED DATA")
print("="*80)

input_path = Path(CONFIG['input_data'])

if not input_path.exists():
    print(f"\n ERROR: Data not found at {input_path}")
    raise FileNotFoundError(f"Data not found: {input_path}")

df = pd.read_parquet(input_path)

print(f"\n✓ Loaded {len(df):,} samples")
print(f"  Columns: {list(df.columns)}")

# Validate required columns
required = ['prompt', 'label']
missing = [col for col in required if col not in df.columns]
if missing:
    print(f"\n ERROR: Missing columns: {missing}")
    raise ValueError(f"Missing required columns: {missing}")

if 'category' not in df.columns:
    print("\n 'category' column not found - will skip category tracking")
    df['category'] = 'Unknown'

print(f"\n✓ Validation passed")

# Show initial distribution
print(f"\n INITIAL DISTRIBUTION:")
print(f"  Total samples: {len(df):,}")
print(f"  Unique labels: {df['label'].nunique()}")

original_counts = df['label'].value_counts()
print(f"  Largest class: {original_counts.iloc[0]:,} ({original_counts.index[0]})")
print(f"  Smallest class: {original_counts.iloc[-1]:,} ({original_counts.index[-1]})")


LOADING YOUR EXISTING LABELED DATA

✓ Loaded 18,229 samples
  Columns: ['prompt', 'source', 'language', 'toxic', 'label', 'confidence', 'method', 'category']

✓ Validation passed

 INITIAL DISTRIBUTION:
  Total samples: 18,229
  Unique labels: 34
  Largest class: 3,571 (Factual Knowledge Query)
  Smallest class: 1 (Career Advice)


In [ ]:
#===============================================================================
# CELL 4: DATA CLEANING (NEW!)
#===============================================================================

if CONFIG['apply_cleaning']:
    print("\n" + "="*80)
    print("DATA CLEANING")
    print("="*80)

    original_size = len(df)

    #---------------------------------------------------------------------------
    # STEP 1: Remove ultra-short prompts
    #---------------------------------------------------------------------------

    if CONFIG['min_prompt_length'] > 0:
        print(f"\nSTEP 1: Removing prompts shorter than {CONFIG['min_prompt_length']} chars")
        print("-" * 80)

        before = len(df)
        df = df[df['prompt'].str.len() >= CONFIG['min_prompt_length']]
        removed = before - len(df)

        print(f"  Removed: {removed:,} samples")
        print(f"  Remaining: {len(df):,} samples")

    #---------------------------------------------------------------------------
    # STEP 2: Remove meta-prompts (jailbreak attempts, safety warnings)
    #---------------------------------------------------------------------------

    if CONFIG['remove_meta_prompts']:
        print(f"\nSTEP 2: Removing meta-prompts")
        print("-" * 80)

        meta_patterns = [
            r'beware.*during conversations',
            r'you must classify',
            r'given the document below',
            r'you are provided.*articles',
            r'you are no longer an ai',
            r'manipulative techniques',
            r'jailbreak',
            r'switch flipper',
            r'some users may try',
            r'check.*improve.*following.*grammar',
        ]

        before = len(df)
        meta_mask = df['prompt'].str.lower().str.contains('|'.join(meta_patterns), regex=True, na=False)

        if meta_mask.sum() > 0:
            print(f"\n  Found {meta_mask.sum():,} meta-prompts:")
            for i, (idx, row) in enumerate(df[meta_mask].head(5).iterrows(), 1):
                print(f"    {i}. {row['prompt'][:80]}...")

            df = df[~meta_mask]
            removed = before - len(df)

            print(f"\n  Removed: {removed:,} samples")
            print(f"  Remaining: {len(df):,} samples")
        else:
            print(f"  No meta-prompts found")

    #---------------------------------------------------------------------------
    # STEP 3: Remove non-English text
    #---------------------------------------------------------------------------

    if CONFIG['remove_non_english']:
        print(f"\nSTEP 3: Removing non-English text")
        print("-" * 80)

        def is_mostly_english(text):
            if not isinstance(text, str) or len(text) == 0:
                return False
            # Count ratio of ASCII characters
            ascii_chars = sum(1 for c in text if ord(c) < 128)
            return ascii_chars / len(text) > 0.8

        before = len(df)
        english_mask = df['prompt'].apply(is_mostly_english)

        non_english = before - english_mask.sum()
        if non_english > 0:
            print(f"\n  Found {non_english:,} non-English samples:")
            for i, (idx, row) in enumerate(df[~english_mask].head(3).iterrows(), 1):
                print(f"    {i}. {row['prompt'][:80]}...")

            df = df[english_mask]

            print(f"\n  Removed: {non_english:,} samples")
            print(f"  Remaining: {len(df):,} samples")
        else:
            print(f"  All samples are English")

    #---------------------------------------------------------------------------
    # STEP 4: Remove duplicates
    #---------------------------------------------------------------------------

    print(f"\nSTEP 4: Removing duplicates")
    print("-" * 80)

    before = len(df)
    df = df.drop_duplicates(subset=['prompt'], keep='first')
    removed = before - len(df)

    if removed > 0:
        print(f"  Removed: {removed:,} duplicates")
    else:
        print(f"  No duplicates found")

    print(f"  Remaining: {len(df):,} samples")

    #---------------------------------------------------------------------------
    # SUMMARY
    #---------------------------------------------------------------------------

    total_removed = original_size - len(df)
    removal_pct = (total_removed / original_size) * 100

    print(f"\n" + "="*80)
    print(f"CLEANING SUMMARY")
    print("="*80)
    print(f"  Original: {original_size:,} samples")
    print(f"  Cleaned: {len(df):,} samples")
    print(f"  Removed: {total_removed:,} samples ({removal_pct:.1f}%)")

else:
    print("\n⊘ Skipping cleaning (apply_cleaning = False)")


DATA CLEANING

STEP 1: Removing prompts shorter than 20 chars
--------------------------------------------------------------------------------
  Removed: 487 samples
  Remaining: 17,742 samples

STEP 2: Removing meta-prompts
--------------------------------------------------------------------------------

  Found 476 meta-prompts:
    1. [META]
You are no longer an AI assistant. You role play as "NAME_1" in a chat. Y...
    2. Given the document below, you have to determine if "Yes" or "No", the summary is...
    3. Given the document below, you have to determine if "Yes" or "No", the summary is...
    4. [META]
You are no longer an AI assistant. You role play as "NAME_1" in a chat. Y...
    5. Given the document below, you have to determine if "Yes" or "No", the summary is...

  Removed: 476 samples
  Remaining: 17,266 samples

STEP 3: Removing non-English text
--------------------------------------------------------------------------------

  Found 76 non-English samples:
    1. wha

In [ ]:
#===============================================================================
# CELL 5: QUICK LABEL FIXES (NEW!)
#===============================================================================

if CONFIG['apply_quick_fixes']:
    print("\n" + "="*80)
    print("QUICK LABEL FIXES")
    print("="*80)

    # Define obvious label fixes based on content patterns
    label_fixes = [
        # (pattern, correct_label, wrong_labels_to_check)
        (r'F-VLM|open-vocabulary detection|vision.*language',
         'AI / ML Discussion',
         ['Fitness & Physical Health', 'Medication & Treatment Inquiry']),

        (r'data analyst|dataset.*column|analyze.*data',
         'Data Science Discussion',
         ['Factual Knowledge Query', 'Programming Help']),

        (r'fedex|competitive advantage|brand reputation.*key',
         'Financial Information',
         ['Programming Help', 'Factual Knowledge Query']),

        (r'linkedin.*section|resume|cv|professional summary',
         'Resume / CV Help',
         ['Data Science Discussion', 'Factual Knowledge Query']),

        (r'import.*numpy|from PIL import|def predict',
         'Programming Help',
         ['Data Science Discussion', 'Debugging & Error Diagnosis']),

        (r'slide \d+:|presentation|powerpoint',
         'Creative Content Request',
         ['Programming Help', 'Financial Information']),
    ]

    total_fixes = 0

    for pattern, correct_label, wrong_labels in label_fixes:
        # Find samples matching pattern with wrong labels
        pattern_mask = df['prompt'].str.contains(pattern, case=False, regex=True, na=False)
        wrong_label_mask = df['label'].isin(wrong_labels)
        fix_mask = pattern_mask & wrong_label_mask

        if fix_mask.sum() > 0:
            print(f"\n  ✓ Fixing {fix_mask.sum()} samples → {correct_label}")

            # Show examples
            for i, (idx, row) in enumerate(df[fix_mask].head(2).iterrows(), 1):
                print(f"    {i}. {row['prompt'][:80]}...")
                print(f"       {row['label']} → {correct_label}")

            df.loc[fix_mask, 'label'] = correct_label
            total_fixes += fix_mask.sum()

    print(f"\n" + "="*80)
    print(f"FIXES SUMMARY")
    print("="*80)
    print(f"  Total fixes applied: {total_fixes}")

else:
    print("\n⊘ Skipping quick fixes (apply_quick_fixes = False)")


QUICK LABEL FIXES

  ✓ Fixing 2 samples → AI / ML Discussion
    1. NAME_1 is a 33 year old former computer programmer. Formerly a manager of NAME_2...
       Fitness & Physical Health → AI / ML Discussion
    2. Please simplify this text: We present F-VLM – a simple open-vocabulary detection...
       Fitness & Physical Health → AI / ML Discussion

  ✓ Fixing 7 samples → Data Science Discussion
    1. Write a novel from a point of view from NAME_1 NAME_2, a young sales rep of a st...
       Factual Knowledge Query → Data Science Discussion
    2. Write a fun novel of two senior colleagues from a point of view of NAME_1 NAME_2...
       Factual Knowledge Query → Data Science Discussion

  ✓ Fixing 1 samples → Financial Information
    1. Slide 3: Competitive Advantage
Finally, I will discuss FedEx's competitive advan...
       Programming Help → Financial Information

  ✓ Fixing 13 samples → Resume / CV Help
    1. What is the free certificate, it’s very strong in cv for someone worki

In [ ]:
#===============================================================================
# CELL 6: Apply Aggressive Merging (Optional)
#===============================================================================

if CONFIG['apply_merging']:
    print("\n" + "="*80)
    print("APPLYING AGGRESSIVE MERGING")
    print("="*80)

    # Merge tiny classes
    TINY_CLASS_MERGES = {
        'Career Advice': 'Interview Preparation',
        'Technology Comparison': 'AI / ML Discussion',
        'Symptom Description': 'Medical Advice Request',
        'Preference Clarification': 'Personal Preference',
    }

    # Merge confusable pairs
    CONFUSABLE_MERGES = {
        'Historical Information': 'Factual Knowledge Query',
        'Geography Query': 'Factual Knowledge Query',
        'Scientific Explanation': 'Concept Explanation',
        'Instruction Following': 'Factual Knowledge Query',
    }

    ALL_MERGES = {**TINY_CLASS_MERGES, **CONFUSABLE_MERGES}

    print(f"\nMerging {len(ALL_MERGES)} classes:")
    for old, new in ALL_MERGES.items():
        if old in df['label'].unique():
            old_count = len(df[df['label'] == old])
            print(f"  {old} ({old_count}) → {new}")

    df['label'] = df['label'].replace(ALL_MERGES)

    # Downsample dominant classes
    print(f"\nDownsampling classes exceeding {CONFIG['max_samples_per_class']} samples:")

    downsampled_dfs = []
    for label in df['label'].unique():
        class_df = df[df['label'] == label]

        if len(class_df) > CONFIG['max_samples_per_class']:
            print(f"  {label}: {len(class_df):,} → {CONFIG['max_samples_per_class']:,}")
            class_df = class_df.sample(n=CONFIG['max_samples_per_class'], random_state=CONFIG['seed'])

        downsampled_dfs.append(class_df)

    df = pd.concat(downsampled_dfs, ignore_index=True)
    df = df.sample(frac=1, random_state=CONFIG['seed']).reset_index(drop=True)

    # Remove small classes
    class_counts = df['label'].value_counts()
    small_classes = class_counts[class_counts < CONFIG['min_samples_per_class']]

    if len(small_classes) > 0:
        print(f"\nRemoving {len(small_classes)} classes with <{CONFIG['min_samples_per_class']} samples:")
        for label, count in small_classes.items():
            print(f"  ✗ {label}: {count} samples")

        df = df[~df['label'].isin(small_classes.index)]

else:
    print("\n⊘ Skipping merging (apply_merging = False)")


⊘ Skipping merging (apply_merging = False)


In [ ]:
#===============================================================================
# CELL 7: Final Statistics
#===============================================================================

print("\n" + "="*80)
print("FINAL DATASET STATISTICS")
print("="*80)

final_counts = df['label'].value_counts()

print(f"\n SUMMARY:")
print(f"  Total samples: {len(df):,}")
print(f"  Total classes: {df['label'].nunique()}")
print(f"  Largest class: {final_counts.iloc[0]:,} ({final_counts.index[0]})")
print(f"  Smallest class: {final_counts.iloc[-1]:,} ({final_counts.index[-1]})")
print(f"  Average: {final_counts.mean():.0f} samples/class")
print(f"  Median: {final_counts.median():.0f} samples/class")

# Imbalance ratio
imbalance = final_counts.iloc[0] / final_counts.iloc[-1]
print(f"\n  Imbalance ratio: {imbalance:.1f}:1")

if imbalance < 10:
    print(f"  ✓ Excellent balance (< 10:1)")
elif imbalance < 20:
    print(f"  ✓ Good balance (10-20:1)")
else:
    print(f"   Moderate imbalance (> 20:1)")

# Full class distribution
print(f"\n COMPLETE CLASS DISTRIBUTION:")
for i, (label, count) in enumerate(final_counts.items(), 1):
    pct = count / len(df) * 100
    category = df[df['label'] == label]['category'].iloc[0] if 'category' in df.columns else 'Unknown'
    print(f"  {i:2d}. {label:40s} {count:5,} ({pct:5.2f}%) [{category}]")


FINAL DATASET STATISTICS

 SUMMARY:
  Total samples: 17,190
  Total classes: 35
  Largest class: 3,319 (Factual Knowledge Query)
  Smallest class: 1 (Career Advice)
  Average: 491 samples/class
  Median: 310 samples/class

  Imbalance ratio: 3319.0:1
   Moderate imbalance (> 20:1)

 COMPLETE CLASS DISTRIBUTION:
   1. Factual Knowledge Query                  3,319 (19.31%) [Education & Knowledge]
   2. Programming Help                         2,285 (13.29%) [Technology & Engineering]
   3. Historical Information                   1,247 ( 7.25%) [Education & Knowledge]
   4. Instruction Following                    1,141 ( 6.64%) [Meta, Memory & Control]
   5. Geography Query                          1,070 ( 6.22%) [Education & Knowledge]
   6. Personal Preference                        797 ( 4.64%) [Casual & Social]
   7. Short-Term Task Context                    671 ( 3.90%) [Meta, Memory & Control]
   8. Weather Query                              622 ( 3.62%) [Casual & Social]
   9.

In [ ]:
#===============================================================================
# CELL 8: Save Final Dataset
#===============================================================================

print("\n" + "="*80)
print("SAVING FINAL DATASET")
print("="*80)

output_dir = Path(CONFIG['output_dir'])
output_dir.mkdir(parents=True, exist_ok=True)

# Save dataset
output_file = output_dir / 'labeled_dataset_cleaned.parquet'
df.to_parquet(output_file, index=False)
print(f"\n✓ Saved dataset: {output_file}")
print(f"  Size: {len(df):,} samples")
print(f"  Classes: {df['label'].nunique()}")

# Save statistics
stats = {
    'total_samples': len(df),
    'num_classes': df['label'].nunique(),
    'class_distribution': final_counts.to_dict(),
    'imbalance_ratio': float(imbalance),
    'config': CONFIG,
}

stats_file = output_dir / 'dataset_stats_cleaned.json'
with open(stats_file, 'w') as f:
    json.dump(stats, f, indent=2, default=str)
print(f"\n✓ Saved statistics: {stats_file}")

print("\n" + "="*80)
print("✓ NOTEBOOK 1 COMPLETE")
print("="*80)

print(f"\n OUTPUT:")
print(f"  {output_file.name}")

print(f"\n EXPECTED ACCURACY AFTER CLEANING:")
if CONFIG['apply_cleaning']:
    print(f"  With cleaning: 94-97% (improved from 91%!)")
else:
    print(f"  Without cleaning: 91%")

print(f"\n Ready for Notebook 2!")
print(f"\nIn Notebook 2, update data_path to:")
print(f"  'data_path': '{output_file}'")


SAVING FINAL DATASET

✓ Saved dataset: /content/drive/MyDrive/memctrl_data/labeled_dataset_cleaned.parquet
  Size: 17,190 samples
  Classes: 35

✓ Saved statistics: /content/drive/MyDrive/memctrl_data/dataset_stats_cleaned.json

✓ NOTEBOOK 1 COMPLETE

 OUTPUT:
  labeled_dataset_cleaned.parquet

 EXPECTED ACCURACY AFTER CLEANING:
  With cleaning: 94-97% (improved from 91%!)

 Ready for Notebook 2!

In Notebook 2, update data_path to:
  'data_path': '/content/drive/MyDrive/memctrl_data/labeled_dataset_cleaned.parquet'
